# P2 CHAIN D — exam reds (wiring + ADT)

Closed-book. Marksheet lens: required behaviour + evidence — not elegance.

**Targets (from Chains A–C remake)**
1. Flask↔sqlite **name contracts** (`request.form` keys, `url_for` endpoints, `LIKE`, **return** redirect)
2. LinkedList of **ids** + **head delete**
3. HashTable + BST **built and used** + correct pre/in/post
4. Flask↔mongo same wiring discipline
5. **LAST sockets visit** — `recv_line` only; after this, do not practise sockets again (A-level P2 hit rate ≈ 0 in TYS; even RI TP is a thin slice)

**Rules**
- Checkpoints encapsulated at the **bottom**. No answer keys in this notebook (`_archive/P2 CHAIN D - answers.ipynb` when marking).
- New template names under `templates/chain_d/` — do **not** reuse Chain A/C key names from memory; open the HTML once to read `name=` / `url_for`.
- Port **5003** (sqlite Flask) and **5004** (mongo Flask).

**DB:** uses `chain_a.db` if present (from Chain A). If missing, run the SETUP cell once.


---
## SETUP — seed `chain_a.db` if needed (not a red; skip if A already seeded)

If checkpoint D1 cannot open a populated DB, run this once. Prefer your Chain A seed if it still works.


In [2]:
# SETUP — run only if chain_a.db missing / empty
import csv, sqlite3, os
from pathlib import Path

if Path("chain_a.db").exists():
    conn = sqlite3.connect("chain_a.db")
    n = conn.execute("SELECT COUNT(*) FROM sqlite_master WHERE type='table'").fetchone()[0]
    conn.close()
    if n >= 2:
        print("chain-_a.db already has tables — skip SETUP")
    else:
        print("re-seed needed — edit this cell or re-run Chain A2")
else:
    print("no chain_a.db — run Chain A1+A2 first, or paste a minimal seed here")


chain-_a.db already has tables — skip SETUP


---
## Part D1 — Flask ↔ sqlite name wiring (15 min)  [Gate 4 red]

Templates: `templates/chain_d/front.html`, `show.html`, `record.html`.

| Route | Endpoint **def name** | Template | Form / params |
|-------|----------------------|----------|----------------|
| `/` GET+POST | `front` | `chain_d/front.html` | POST: `q_session`, `q_name` → **return** `redirect(url_for("show", ...))` |
| `/show/<q_session>/<q_name>` | `show` | `chain_d/show.html` | JOIN; `Member.name LIKE ?` with wildcards **in the tuple**; pass `rows`, `q_session`, `q_name` |
| `/record` GET+POST | `record` | `chain_d/record.html` | POST: `in_mid`, `in_date`, `in_session` → INSERT `?` → **return** `redirect(url_for("front"))` |

Also: `get_db()` + `row_factory`. `app.run(port=5003)`.

**Evidence:** structural checkpoint + one manual search + one insert.


In [4]:
# Part D1 — YOUR ATTEMPT
from flask import Flask, render_template, request, redirect, url_for
import sqlite3

app = Flask(__name__)

# get_db, front, show, record, __main__ port 5003

def get_db():
    conn = sqlite3.connect("chain_a.db")
    conn.row_factory = sqlite3.Row
    return conn

ac = ['GET', 'POST']

@app.route('/', methods = ac)
def front():
    error = None
    if request.method == 'POST':
        q_session = request.form.get('q_session','').strip()
        q_name = request.form.get('q_name','').strip()
        if q_session and q_name:
            return redirect(url_for('show', q_session = q_session, q_name = q_name))
        error = 'Please fill in all fields'
    return render_template('chain_d/front.html', error=error)

@app.route('/show/<q_session>/<q_name>', methods = ac)
def show(q_session, q_name):
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
    SELECT Member.name, Attendance.date, Attendance.session
    FROM Member INNER JOIN Attendance ON 
    Member.member_id = Attendance.member_id
    WHERE Member.name LIKE ? AND Attendance.session = ?
    """, (f"%{q_name}%",q_session))
    rows = cur.fetchall()
    conn.close()
    return render_template('chain_d/show.html', q_session = q_session, q_name = q_name, rows = rows)

@app.route('/record', methods = ac)
def record():
    error = None
    if request.method == 'POST':
        in_mid = request.form.get('in_mid', '')
        in_date = request.form.get('in_date', '')
        in_session = request.form.get('in_session','')
        if in_mid and in_date and in_session:
            conn = get_db()
            cur = conn.cursor()
            cur.execute("""
            INSERT OR IGNORE INTO Attendance (member_id, date, session)
            VALUES (?,?,?)
            """, (in_mid, in_date, in_session))
            conn.commit()
            conn.close()
            return redirect(url_for('record'))
        error = 'Please fill in all fields'
    return render_template('chain_d/record.html', error = error)

if __name__ == '__main__':
    app.run(port=5003)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5003
Press CTRL+C to quit
127.0.0.1 - - [27/Sep/2026 15:49:58] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [27/Sep/2026 15:49:58] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [27/Sep/2026 15:50:01] "POST / HTTP/1.1" 302 -
127.0.0.1 - - [27/Sep/2026 15:50:01] "GET /show/Training/is HTTP/1.1" 200 -
127.0.0.1 - - [27/Sep/2026 15:50:01] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [27/Sep/2026 15:50:03] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [27/Sep/2026 15:50:03] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [27/Sep/2026 15:57:22] "POST / HTTP/1.1" 302 -
127.0.0.1 - - [27/Sep/2026 15:57:22] "GET /show/Training/lim HTTP/1.1" 200 -
127.0.0.1 - - [27/Sep/2026 15:57:22] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [27/Sep/2026 15:57:29] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [27/Sep/2026 15:57:29] "GET /static/style.css HTTP/1.1" 304 -


---
## Part D2 — LinkedList of ids + head delete (12 min)  [Gate 3 red]

Build `Node` + `LinkedList` (`is_empty`, `insert_end`, `search` → Node/None, `delete` **including head**).

Driver (required evidence):
```text
ids = [10, 20, 30, 40, 50]
# insert_end each
# delete head (10)
# search(30) found; search(-1) is None; delete(-1) no crash
# walk remaining ids somehow (helper or prints)
```

Store **integers**, not objects. `is_empty` must mean “no nodes.”


In [1]:
# Part D2 — YOUR ATTEMPT

ids = [10, 20, 30, 40, 50]
catalogue = None

class Node:
    def __init__(self, data, right = None):
        self.__data = data
        self.__right = right
        
    def get_data(self):
        return self.__data
    
    def set_data(self, data):
        self.__data = data
        
    def get_right(self):
        return self.__right
    
    def set_right(self, right):
        self.__right = right
        
class LinkedList:
    def __init__(self, start = None):
        self.__start = start
        
    def is_empty(self):
        return not bool(self.__start)
    
    def insert_end(self, data):
        new = Node(data)
        if self.is_empty():
            self.__start = new
            return True
        current = self.__start
        while current.get_right():
            current = current.get_right()
        current.set_right(new)
        return True
        
    def search(self, data):
        current = self.__start
        while current: 
            previous = current
            current = current.get_right()
            if previous.get_data() == data:
                return previous
        return current
    
    def delete(self, data):
        if self.is_empty():
            return False
        elif self.__start.get_data() == data:
            self.__start = self.__start.get_right()
            return True
        current = self.__start
        while current:
            pevious = current
            if current.get_data() == data:
                previous.set_right(current.get_right())
                return True
            current = current.get_right()
        return False
    
    def display(self):
        current = self.__start
        fin = ''
        while current:
            fin += f'->{current.get_data()}'
            current = current.get_right()
        return fin

catalogue = LinkedList()
for i in ids:
    catalogue.insert_end(i)
    print(i)
print('\n')    
catalogue.delete(10)
catalogue.search(30)
catalogue.search(-1)
catalogue.delete(-1)
catalogue.display()

10
20
30
40
50




'->20->30->40->50'

---
## Part D3 — HashTable + BST from catalogue (12 min)  [Gate 3 red]

Depends on **D2** working.

1. `HashTable(7)` separate chaining with `[LinkedList() for _ in range(7)]`. `insert` / `search` → bool. Insert **every id still in catalogue** after the head delete.
2. `TreeNode` + `BST`: `insert` (`<=` left), `get_root`, `in_order` / `pre_order` / `post_order` each taking a **node**.  
   **pre/post must call themselves** on children (not `in_order`).
3. Print in-order. Keep `ht` and `bst` assigned (not left as `None`).


In [5]:
# Part D3 — YOUR ATTEMPT

ht = None
bst = None

class HashTable:
    def __init__(self, N):
        self.__size = N
        self.__arr = [LinkedList() for _ in range(N)]
    
    def hash_fn(self, data):
        if data:
            return int(data) % self.__size
    
    def search(self, data):
        if data:
            return self.__arr[self.hash_fn(data)].search(data)
    
    def insert(self, data):
        if data:
            return self.__arr[self.hash_fn(data)].insert_end(data)
    
    def display(self):
        for n in range(self.__size):
            print(f"{n} {self.__arr[n].display()}")

ht = HashTable(7)
#extract info from catalogue
lis = catalogue.display().split('->')
lis.pop(0)
for i in lis:
    ht.insert(i)
ht.display()
print('\n')

class TreeNode:
    def __init__(self, data):
        self.__data = data
        self.__right = None
        self.__left = None
        
    def get_data(self):
        return self.__data
    
    def set_data(self, data):
        self.__data = data
        
    def get_right(self):
        return self.__right
    
    def set_right(self, right):
        self.__right = right
        
    def get_left(self):
        return self.__left
    
    def set_left(self, left):
        self.__left  = left
        
class BST:
    def __init__(self, root = None):
        self.__root = root
        
    def get_root(self):
        return self.__root
        
    def insert(self, data):
        new = TreeNode(data)
        if self.__root is None:
            self.__root = new
            return
        current = self.__root
        while current:
            if data <= self.__root.get_data():
                if current.get_left() is None:
                    current.set_left(new)
                    return
                current = current.get_left()
            else:
                if current.get_right() is None:
                    current.set_right(new)
                    return
                current = current.get_right()
                    
    def in_order(self, node):
        if node is not None:
            self.in_order(node.get_left())
            print(node.get_data())
            self.in_order(node.get_right())
            
    def pre_order(self, node):
        if node is not None:
            print(node.get_data())
            self.pre_order(node.get_left())
            self.pre_order(node.get_right())
            
    def post_order(self, node):
        if node is not None:
            self.post_order(node.get_left())
            self.post_order(node.get_right())
            print(node.get_data())
    
bst = BST()
for i in lis:
    bst.insert(i)

bst.in_order(bst.get_root())
            

0 
1 ->50
2 ->30
3 
4 
5 ->40
6 ->20


20
30
40
50


---
## Part D4 — Flask ↔ mongo name wiring (15 min)  [Gate 6 web slice]

Requires mongod. Seed collection yourself in this part (or reuse `chain_c_db.staff` if still populated).

Templates: `templates/chain_d/lobby.html`, `listed.html`, `onboard.html`.

| Route | Endpoint | Template | Notes |
|-------|----------|----------|-------|
| `/` GET+POST | `lobby` | `chain_d/lobby.html` | form `f_dept`, `f_surname` → **return** redirect to `listed` |
| `/listed/<f_dept>/<f_surname>` | `listed` | `chain_d/listed.html` | `render_template("chain_d/listed.html", people=...)` — **not** `url_for` as template path |
| `/onboard` GET+POST | `onboard` | `chain_d/onboard.html` | methods include POST; form `f_sid`, `f_surname`, `f_given`, `f_dept`; `insert_one`; **return** `redirect(url_for("lobby"))` |

`get_mongo()` → staff collection. Port **5004**.

**Evidence:** endpoints registered + one find + one onboard if mongod up.


In [ ]:
# Part D4 — YOUR ATTEMPT
from flask import Flask, render_template, request, redirect, url_for
from pymongo import MongoClient

app_mongo = Flask("chain_d_mongo")

# get_mongo, lobby, listed, onboard, __main__ port 5004
def get_mongo():
    db = pymongo.MongoClient("mongodb://127.0.0.1:27017")["chain_c_db"]["staff"]
    return db

#we pretend that the database has been populated
ac = ['GET', 'POST']

@app.route('/', methods = ac)
def lobby():
    error = None
    if request.method == 'POST':
        f_dept = request.form.get('f_dept', '')
        f_surname = request.form.get('f_surname','')
        if f_dept and f_surname:
            return redirect(url_for('listed'), f_dept = f_dept, f_surname = f_surname)
        error = 'Please fill in all fields'
    return render_template('chain_d/lobby.html', error = error)

@app.route('/listed/<f_dept>/<f_surname>', methods = ac)
def listed(f_dept, f_surname):
    people = get_mongo().find({'department': f_dept, 'surname': f_surname}, {'_id': 0, 'surname': 1, 'given': 1, 'department': 1})
    return render_template('chain_d/listed/html', people = people)

@app.route('/onboard', methods = ac)
def onboard():
    error = None
    if request.method == 'POST':
        f_sid = request.form.get('f_sid','')
        f_surname = request.form.get('f_surname','')
        f_given = request.form.get('f_given','')
        f_dept = request.form.get('f_dept','')
        if f_sid and f_surname and f_given and f_dept:
            headers = ['staff_id', 'surname', 'given', 'department']
            get_mongo().insert_one(dict(zip(headers, [f_sid, f_surname, f_given, f_dept])))
            return redirect(url_for('lobby'))
        error = 'Please fill in all fields'
    return render_template('chain_d/onboard.html', error = error)

---
## Part D5 — LAST sockets visit (5 min)  [then stop sockets]

Write `recv_line(sock)` only: loop `recv(1024)` until `b"\n"` (or peer closed); decode utf-8; strip.

Comment server/client spines in one line each if you want — **do not** build a full game protocol.

After checkpoint: **sockets leave the practice rotation** (TYS P2 2020–2025: 0 hits in this repo; RI TP still a thin mark cluster).


In [ ]:
# Part D5 — YOUR ATTEMPT — last sockets practice

def recv_line(sock):
    data = b''
    while b'\n' not in data:
        chunk = sock.recv(1024)
        if not chunk:
            break
        data += chunk
    return data.decode.strip()

#server code
server = socket.socket()
server.bind(('127.0.0.1', 6767))
server.listen()
client, address = server.accept()

while True:
    data = recv_line(client)
    #loop
    client.sendall(b'message\n')
    
client.close()
server.close()

#client code
client = socket.socket()
client.connect(('127.0.0.1', 6767))
done = False

while not done:
    client.sendall(f'{input('type something')}\n'.encode())
    #conditions
    done = bool(client.recv_line(client))
    
client.close()


---
# Checkpoints (encapsulated)

Edit `PARTS`, then run. Not a writing guide.


In [ ]:
# CHECKPOINTS — Chain D

PARTS = ("D1", "D2", "D3", "D4", "D5")


def _check_d1():
    g = globals()
    for name in ("app", "get_db", "front", "show", "record"):
        if name not in g:
            return False, f"missing {name}"
    rules = {rule.rule: rule.endpoint for rule in app.url_map.iter_rules() if rule.endpoint != "static"}
    if rules.get("/") != "front" or rules.get("/record") != "record":
        return False, f"route map unexpected: {rules}"
    if "show" not in rules.values():
        return False, "show endpoint missing"
    return True, "ok (manual: port 5003 search + insert)"


def _check_d2():
    if catalogue is None:
        return False, "catalogue missing"
    if not catalogue.is_empty() and catalogue.search(10) is not None:
        # head 10 should be gone
        return False, "id 10 still present — head delete?"
    if catalogue.search(30) is None:
        return False, "30 missing"
    if catalogue.search(-1) is not None:
        return False, "search(-1) should be None"
    # is_empty semantics: empty list → True
    empty = type(catalogue)()
    if empty.is_empty() is not True:
        return False, "is_empty() wrong on empty list"
    return True, "ok"


def _check_d3():
    if ht is None or bst is None:
        return False, "ht or bst still None — not constructed"
    if not ht.search(30):
        return False, "hash miss on 30"
    if not hasattr(bst, "get_root") or bst.get_root() is None:
        return False, "bst root missing"
    # pre_order must not be an alias bug: call and ensure it runs
    bst.in_order(bst.get_root())
    return True, "ok"


def _check_d4():
    g = globals()
    app_m = g.get("app_mongo") or g.get("app")
    if app_m is None:
        return False, "app_mongo/app missing"
    for name in ("get_mongo", "lobby", "listed", "onboard"):
        if name not in g:
            return False, f"missing {name}"
    eps = {rule.endpoint for rule in app_m.url_map.iter_rules()}
    for ep in ("lobby", "listed", "onboard"):
        if ep not in eps:
            return False, f"endpoint {ep} not registered"
    return True, "ok (manual: port 5004 if mongod up)"


def _check_d5():
    import inspect
    if "recv_line" not in globals() or not callable(recv_line):
        return False, "recv_line missing"
    src = inspect.getsource(recv_line)
    if "pass" in src and "recv" not in src.replace("recv_line", ""):
        return False, "recv_line body still empty"
    if "recv" not in src:
        return False, "recv_line should call sock.recv"
    return True, "ok — sockets practice ends after this"


_CHECKS = {
    "D1": _check_d1,
    "D2": _check_d2,
    "D3": _check_d3,
    "D4": _check_d4,
    "D5": _check_d5,
}

for part in PARTS:
    try:
        ok, detail = _CHECKS[part]()
    except Exception as e:
        ok, detail = False, str(e)
    print(f"{part}: {'PASS' if ok else 'FAIL — ' + detail}")
